# Notebook 01, Validation numérique du PLNE de rendez-vous ambulatoires

**Sections 0–7** miroir des sections 11.1–11.4 du manuscrit (COR/EJOR).
Toutes les durées sont en **minutes**. Ne pas modifier la formulation ; toute simplification computationnelle est signalée en tête de section.

In [1]:
# ── Décisions des auteurs (paramètres papermill) ─────────────────────────────
DURATION_METHOD = "kmedians"    # B3 ; "kmeans" en ablation
DURATION_STAT   = "median"      # s_A, s_B = centroïdes K-médianes
C_O_BASE        = 0.0           # V9 : pas d'overtime, comme le manuscrit (éq. 1, 8, 9) ; 0,5 pour l'extension §8.5
SLOT_LENGTHS    = (30, 60)      # B4 : assertion s_A <= max(SLOT_LENGTHS), sinon arrêt
SPLIT_MODE      = "stratified"  # "patient_group" en ablation
RANDOM_STATE    = 0
SMOKE           = False         # True : 1 % des données, budgets minimaux
PLNE_PARAMS_PATH = "data/inference/plne_parameters.json"  # exporté par data_driven_plan.ipynb
HOLDOUT_SCORES_PATH = None  # holdout_scores.parquet du même run (q_i, groupes, no-show) ; requis
CAMPAIGN_OVERRIDES = {}     # ex. {"R": 10, "time_limit": 600.0} ; voir ilp_campaign.CampaignConfig
N_WORKERS = None            # None : un processus par cœur physique
CACHE_DIR = "results/ilp_cache"  # solutions par (cellule, graine, méthode, configuration)


In [2]:
# Parameters
CACHE_DIR = "results/ilp_cache_serveall"
HOLDOUT_SCORES_PATH = "results/run_full_20261030/holdout_scores.parquet"


In [3]:
# ── 0. Environnement ──────────────────────────────────────────────────────────
import sys, platform, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from dataclasses import dataclass, field
from typing import Optional
import joblib
import os

# Cœur du modèle PLNE partagé avec data_driven_plan.py (racine du projet),
# garantit que ce notebook et data_driven_plan.ipynb résolvent EXACTEMENT
# le même modèle (pas de duplication de solve_ilp/Instance/Config). Ce
# notebook est exécuté avec le répertoire du projet comme répertoire courant.
sys.path.insert(0, os.getcwd())
for _d in ("figures", "tables", "results"):   # outputs; absent from a fresh clone (not versioned)
    os.makedirs(_d, exist_ok=True)
from data_driven_plan import (
    Instance, Config, Solution, compute_coefficients, solve_ilp, SOLVER_NAME,
    objective_latex, validate_decisions,
    eval_slot_exact, eval_day_exact, evaluate_planning_exact,
    simulate_day, simulate_instance, draw_scenarios,
)
import pyomo.environ as pyo

# Figures of the article: data_driven_plan sets the large fonts of the training
# notebook (15 pt labels), which overlap or get cut in multi-panel figures sized
# for a printed page. Smaller fonts here, figures sized close to their print width.
plt.rcParams.update({"font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
                     "legend.title_fontsize": 7, "lines.markersize": 4, "lines.linewidth": 1.4})

DECISIONS = validate_decisions(DURATION_METHOD, DURATION_STAT, C_O_BASE, SLOT_LENGTHS,
                               SPLIT_MODE, RANDOM_STATE, SMOKE)
for _k, _v in DECISIONS.items():
    print(f"{_k:<16} {_v}")

RNG_SEED = RANDOM_STATE
_RNG = np.random.default_rng(RNG_SEED)

print(f"Python  : {sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
import pyomo; print(f"Pyomo   : {pyomo.__version__}")
import psutil
def _cpu_model():
    try:
        with open("/proc/cpuinfo") as fh:
            return next(l.split(":", 1)[1].strip() for l in fh if l.startswith("model name"))
    except (OSError, StopIteration):
        return platform.processor() or "unknown"
CPU_MODEL = _cpu_model()
print(f"Platform: {CPU_MODEL} | {platform.platform()}")
print(f"CPU     : {psutil.cpu_count(logical=False)} cœurs physiques, {psutil.cpu_count()} logiques")
print(f"RAM     : {psutil.virtual_memory().total / 2**30:.1f} Go")
print(f"Solveur : {SOLVER_NAME} ✓ (détecté par data_driven_plan.init_solver)")


xgboost 3.2.0 | imbalanced-learn 0.14.2
DURATION_METHOD  kmedians
DURATION_STAT    median
SPLIT_MODE       stratified
C_O_BASE         0.0
SLOT_LENGTHS     (30.0, 60.0)
RANDOM_STATE     0
SMOKE            False
Python  : 3.13.13
NumPy   : 2.4.6
Pandas  : 3.0.3
Pyomo   : 6.10.1
Platform: Intel(R) Core(TM) i7-6700HQ CPU @ 2.60GHz | Linux-7.0.0-38-generic-x86_64-with-glibc2.39
CPU     : 4 cœurs physiques, 8 logiques
RAM     : 15.5 Go
Solveur : appsi_highs ✓ (détecté par data_driven_plan.init_solver)


## 0.2 Paramètres issus du pipeline de prédiction

Les durées $s_A$, $s_B$, la part du groupe long $\pi_A$ et le taux de no-show sont lus **uniquement** dans `plne_parameters.json`, exporté par `data_driven_plan.ipynb` (K-médianes exactes ajustées sur l'apprentissage seul, provenance incluse). Aucun recalcul ici, aucune valeur par défaut : un champ absent ou hors domaine arrête le notebook (audit B4). Contrôles : $0{,}05 < \pi_A < 0{,}5$, $s_A > s_B > 0$, $s_A \le \max$(`SLOT_LENGTHS`).

In [4]:
# ── 0.2 Lecture et contrôle de plne_parameters.json ──────────────────────────
from data_driven_plan import load_plne_parameters

PLNE = load_plne_parameters(PLNE_PARAMS_PATH, SLOT_LENGTHS)
s_A, s_B = float(PLNE["s_A"]), float(PLNE["s_B"])
pi_A = float(PLNE["pi_A_actual"])
L_SHORT, L_LONG = (float(v) for v in sorted(SLOT_LENGTHS))
if PLNE["duration_method"] != DURATION_METHOD or PLNE["duration_stat"] != DURATION_STAT:
    raise ValueError(f"plne_parameters.json built with {PLNE['duration_method']}/{PLNE['duration_stat']}, "
                     f"notebook expects {DURATION_METHOD}/{DURATION_STAT}")

print(f"Source           : {PLNE['source_run']} (commit {PLNE['git_commit']}, {PLNE['created']})")
print(f"Méthode          : {PLNE['duration_method']} / {PLNE['duration_stat']}, coupure {PLNE['cutoff_minutes']:.1f} min")
print(f"s_A, s_B         : {s_A:.2f} min, {s_B:.2f} min   (créneaux {L_SHORT:g} et {L_LONG:g} min)")
print(f"pi_A réel / prédit : {pi_A:.4f} / {PLNE['pi_A_predicted']:.4f}")
print(f"Taux de no-show  : {PLNE['noshow_rate']:.4f}")
print(f"Erreur du classifieur de groupe (holdout) : {PLNE['group_error_rate']:.4f}")
if PLNE["smoke"]:
    print("ATTENTION : paramètres d'un smoke test (1 % des patients), pas de résultats publiables.")


Source           : run_full_20261030 (commit 8615e6564b67321c11e2ee6208425014f15d0c85, 2026-10-31T03:17:35)
Méthode          : kmedians / median, coupure 21.5 min
s_A, s_B         : 36.00 min, 7.00 min   (créneaux 30 et 60 min)
pi_A réel / prédit : 0.3463 / 0.3474
Taux de no-show  : 0.0317
Erreur du classifieur de groupe (holdout) : 0.3323


## 1. Modèle

`Instance`, `Config`, `Solution`, `compute_coefficients` et `solve_ilp` sont importés de **data_driven_plan.py** (racine du projet), voir la cellule d'environnement ci-dessus. Ce cœur de modèle est ainsi partagé, à l'identique, avec `data_driven_plan.ipynb` (section 7).

**Tous les patients sont planifiés** (`Config.serve_all = True`) : la contrainte C1 est une égalité, $\sum_{d,t} x_{idt} = 1$. Avec C1 en inégalité, ne pas planifier un patient ne coûtait rien dans l'objectif, et l'optimum refusait les patients dont le double ajoutait plus d'attente qu'il ne retirait d'inactivité (12,5 patients servis sur 16 en moyenne à la charge 2,0). L'égalité exige $n \le 2S$ ; sinon `solve_ilp` s'arrête avec une erreur explicite.

In [5]:
# ── 1.1-1.3 Dataclasses + compute_coefficients + solve_ilp ───────────────────
# (importés de data_driven_plan, voir cellule 0 ; aucune redéfinition ici)
print("Instance/Config/Solution/compute_coefficients/solve_ilp importés depuis data_driven_plan.py")


Instance/Config/Solution/compute_coefficients/solve_ilp importés depuis data_driven_plan.py


## 2. Tests d'exactitude

> Tous ces tests doivent passer **avant** toute expérience.

In [6]:
# ── 2.1 Ré-énumération Bernoulli (≤ 4 scénarios par créneau) ─────────────────
# eval_slot_exact (énumération des 2^k scénarios d'un créneau) est importé de
# data_driven_plan : une seule implémentation pour les tests, les benchmarks et le module.
def test_bernoulli_enum(inst: Instance, sol: Solution, tol=1e-9):
    # Compare E[Idle/Wait/OT] en minutes brutes (formule vs énumération Bernoulli).
    coeffs = sol.coeffs
    a, delta, w, o, gamma = (coeffs[k] for k in ("a","delta","w","o","gamma"))
    err_max = 0.0
    for (d,t), pats in sol.planning.items():
        L = inst.L[d][t]
        ei, ew, eo = eval_slot_exact(pats, inst.q, inst.s, L)   # minutes, non pondérées
        if len(pats) == 1:
            i = pats[0]
            mi = L - a[i,d,t]    # E[Idle] brut
            mw = 0.0
            mo = o[i,d,t]         # E[OT] brut
        else:
            pi, pj = (pats[0], pats[1]) if pats[0] < pats[1] else (pats[1], pats[0])
            mi = L - a[pi,d,t] - a[pj,d,t] - delta[pi,pj,d,t]
            mw = w[pi,pj]
            mo = o[pi,d,t] + o[pj,d,t] + gamma[pi,pj,d,t]
        err = max(abs(ei - mi), abs(ew - mw), abs(eo - mo))
        err_max = max(err_max, err)
        assert err < tol, f"Créneau ({d},{t}) pats={pats}: err={err:.2e}"
    return err_max


In [7]:
# ── 2.2 Monte Carlo sur un créneau double ─────────────────────────────────────
def test_montecarlo(q1=0.8, q2=0.7, s1=20.0, s2=30.0, L=40.0, M=1_000_000, seed=0):
    rng = np.random.default_rng(seed)
    show1 = rng.random(M) < q1
    show2 = rng.random(M) < q2
    # SPT : s1 < s2 → i1 first
    idle = np.zeros(M); wait = np.zeros(M); ot = np.zeros(M)
    # Both show
    both = show1 & show2
    svc_both = s1 + s2
    idle[both] = np.maximum(0, L - svc_both)
    wait[both] = s1    # j waits s1
    ot[both]   = np.maximum(0, svc_both - L)
    # Only 1
    only1 = show1 & ~show2
    idle[only1] = np.maximum(0, L - s1)
    ot[only1]   = np.maximum(0, s1 - L)
    # Only 2
    only2 = ~show1 & show2
    idle[only2] = np.maximum(0, L - s2)
    ot[only2]   = np.maximum(0, s2 - L)
    # Neither
    neither = ~show1 & ~show2
    idle[neither] = L

    mc_idle, mc_wait, mc_ot = idle.mean(), wait.mean(), ot.mean()

    # Closed-form
    cf_idle = L - q1*min(s1,L) - q2*min(s2,L) - q1*q2*(min(s1+s2,L)-min(s1,L)-min(s2,L))
    cf_wait = q1*q2*min(s1,s2)
    cf_ot   = q1*max(0,s1-L) + q2*max(0,s2-L) + q1*q2*(min(s1,L)+min(s2,L)-min(s1+s2,L))

    print(f"Monte Carlo vs Formule fermée (M={M:,}) :")
    print(f"  E[Idle] MC={mc_idle:.5f}  CF={cf_idle:.5f}  err={abs(mc_idle-cf_idle):.2e}")
    print(f"  E[Wait] MC={mc_wait:.5f}  CF={cf_wait:.5f}  err={abs(mc_wait-cf_wait):.2e}")
    print(f"  E[OT]   MC={mc_ot:.5f}  CF={cf_ot:.5f}  err={abs(mc_ot-cf_ot):.2e}")
    # Tolérance = 5×SE (SE_idle ≈ 0.012, SE_wait ≈ 0.010, SE_OT ≈ 0.005 pour M=1e6)
    assert abs(mc_idle - cf_idle) < 0.05, f"MC Idle diverge: {abs(mc_idle-cf_idle):.4f}"
    assert abs(mc_wait - cf_wait) < 0.05, f"MC Wait diverge: {abs(mc_wait-cf_wait):.4f}"
    assert abs(mc_ot   - cf_ot)   < 0.03, f"MC OT diverge: {abs(mc_ot-cf_ot):.4f}"
    print("  ✓ Monte Carlo converge vers les formules fermées.")

test_montecarlo()


Monte Carlo vs Formule fermée (M=1,000,000) :
  E[Idle] MC=8.59374  CF=8.60000  err=6.26e-03
  E[Wait] MC=11.19576  CF=11.20000  err=4.24e-03
  E[OT]   MC=5.59788  CF=5.60000  err=2.12e-03
  ✓ Monte Carlo converge vers les formules fermées.


In [8]:
# ── 2.3 Proposition 1 : z continus et z binaires, même optimum ───────────────
# Correction P4 : la proposition n'est mise à l'épreuve que si l'optimum contient
# au moins une paire saturée (delta != 0 ou gamma != 0). Instances construites
# pour cela : patients de 36 et 30 min dans des créneaux de 60 min (36 + 30 > 60).
def _saturated_instance(seed):
    rng = np.random.default_rng(seed)
    n = 4    # tous servis : au plus deux patients par créneau
    s = np.where(np.arange(n) % 2 == 0, 36.0, 30.0)
    q = rng.uniform(0.6, 0.95, n)
    return Instance(n=n, D=1, T=[2], L=[[60.0, 60.0]], s=s, q=q, c_I=1.0, c_W=0.5, c_O=C_O_BASE)

def test_z_integrality(n_instances=10):
    print("Proposition 1 (z continu vs z binaire) sur paires saturées :")
    for k in range(n_instances):
        inst = _saturated_instance(k)
        cont = solve_ilp(inst, Config(z_binary=False, require_optimal=True))
        binary = solve_ilp(inst, Config(z_binary=True, require_optimal=True))
        dl, gm = cont.coeffs["delta"], cont.coeffs["gamma"]
        n_sat = sum(1 for (d, t), p in cont.planning.items() if len(p) == 2
                    and max(abs(dl[min(p), max(p), d, t]), abs(gm[min(p), max(p), d, t])) > 1e-9)
        assert n_sat >= 1, f"instance {k} : aucune paire saturée à l'optimum"
        diff = abs(cont.Z_complete - binary.Z_complete)
        assert diff < 1e-6 * max(1.0, abs(binary.Z_complete)), f"Proposition 1 violée : diff={diff}"
        assert cont.options["fractional_z"] == 0
        print(f"  inst {k}: paires saturées={n_sat}, Z_cont={cont.Z_complete:.6f}, "
              f"Z_bin={binary.Z_complete:.6f}, diff={diff:.1e}  OK")
    print("Proposition 1 vérifiée.")

test_z_integrality()


Proposition 1 (z continu vs z binaire) sur paires saturées :


  inst 0: paires saturées=2, Z_cont=48.679575, Z_bin=48.679575, diff=0.0e+00  OK


  inst 1: paires saturées=2, Z_cont=40.583445, Z_bin=40.583445, diff=0.0e+00  OK


  inst 2: paires saturées=2, Z_cont=45.124719, Z_bin=45.124719, diff=0.0e+00  OK


  inst 3: paires saturées=2, Z_cont=44.284954, Z_bin=44.284954, diff=0.0e+00  OK


  inst 4: paires saturées=2, Z_cont=38.038156, Z_bin=38.038156, diff=0.0e+00  OK


  inst 5: paires saturées=2, Z_cont=40.108544, Z_bin=40.108544, diff=0.0e+00  OK


  inst 6: paires saturées=2, Z_cont=44.949359, Z_bin=44.949359, diff=0.0e+00  OK


  inst 7: paires saturées=2, Z_cont=39.504220, Z_bin=39.504220, diff=0.0e+00  OK


  inst 8: paires saturées=2, Z_cont=41.289805, Z_bin=41.289805, diff=0.0e+00  OK


  inst 9: paires saturées=2, Z_cont=39.219499, Z_bin=39.219499, diff=0.0e+00  OK
Proposition 1 vérifiée.


In [9]:
# ── 2.4 Dominance SPT par énumération exacte ─────────────────────────────────
# Correction P4 : l'ancien test comparait q_i q_j min(s) à q_i q_j max(s), une
# tautologie. On évalue ici le même créneau sous les deux ordres avec eval_slot_exact.
def test_spt_dominance(n_pairs=500, seed=3):
    rng = np.random.default_rng(seed)
    for _ in range(n_pairs):
        q = rng.uniform(0.3, 0.99, 2)
        s = rng.uniform(5, 80, 2)
        L = float(rng.choice(SLOT_LENGTHS))
        long_first = [int(np.argmax(s)), int(np.argmin(s))]
        spt = eval_slot_exact(long_first, q, s, L, spt=True)
        lpt = eval_slot_exact(long_first, q, s, L, spt=False)
        assert spt[1] <= lpt[1] + 1e-12                       # attente
        assert abs(spt[0] - lpt[0]) < 1e-9 and abs(spt[2] - lpt[2]) < 1e-9  # idle, overtime
    print(f"Dominance SPT vérifiée sur {n_pairs} paires (attente plus faible, idle et overtime inchangés).")

test_spt_dominance()
print("\n=== Tous les tests d'exactitude sont passés ===")


Dominance SPT vérifiée sur 500 paires (attente plus faible, idle et overtime inchangés).

=== Tous les tests d'exactitude sont passés ===


## 3. Exemple illustratif, Table 1

Instance : n=7, D=1, 4 créneaux (2×30, 2×60 min) ; patients 0 à 2 du groupe A, 3 à 6 du groupe B ; q=[0,90 ; 0,85 ; … ; 0,60] ; c_I=1, c_W=0,5, c_O=0,5 (coûts de la Table 1 du manuscrit, fixés ici indépendamment de `C_O_BASE`). L'ancienne instance avait 10 patients ; son optimum laissait de côté les patients 7 à 9 (q = 0,55 ; 0,50 ; 0,45), ce que la contrainte « tous servis » interdit (10 > 2 × 4). Sans eux, l'optimum et $Z^\star$ sont inchangés.

Correction rec. 5 : l'exemple ne dépend plus des données. Variante **publiée** : $s_A=43$, $s_B=8$ min, $c_O=0{,}5$ (coûts de la Table 1 du manuscrit), doit redonner $Z^\star = 70{,}82$. Variante **historique** : mêmes durées, $c_O$ = `C_O_BASE` ; avec $c_O = 0$ (modèle du manuscrit), $Z^\star = 65{,}62$, valeur à reporter dans le manuscrit. Variante **données** : $s_A$, $s_B$ de `plne_parameters.json`, à titre d'illustration.

In [10]:
# ── 3.1 Construction et résolution des deux variantes ───────────────────────
ILLUS_Q = np.array([0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60])
ILLUS_L = [[30.0, 30.0, 60.0, 60.0]]
ILLUS_GROUPS = np.array(["A"] * 3 + ["B"] * 4)

def illustrative_instance(sA, sB, cO):
    return Instance(n=len(ILLUS_Q), D=1, T=[4], L=ILLUS_L, s=np.where(ILLUS_GROUPS == "A", sA, sB),
                    q=ILLUS_Q, c_I=1.0, c_W=0.5, c_O=cO)

# "publie" : coûts de la Table 1 du manuscrit (c_O = 0,5), contrôle de non-régression de 70,82.
# "historique" et "donnees" : coût d'overtime C_O_BASE (0 : modèle du manuscrit, sans overtime).
ILLUS = {}
for _tag, (_sA, _sB, _cO) in {"publie": (43.0, 8.0, 0.5), "historique": (43.0, 8.0, C_O_BASE),
                              "donnees": (s_A, s_B, C_O_BASE)}.items():
    _inst = illustrative_instance(_sA, _sB, _cO)
    _sol = solve_ilp(_inst, Config(require_optimal=True))
    ILLUS[_tag] = (_inst, _sol)
    print(f"[{_tag}] s_A={_sA:.2f}, s_B={_sB:.2f} : statut {_sol.status}, gap {_sol.gap_complete:.1e}, "
          f"Z* = {_sol.Z_complete:.4f}  (E[Idle]={_sol.E_idle:.3f}, E[Wait]={_sol.E_wait:.3f}, "
          f"E[OT]={_sol.E_OT:.3f}), servis {_sol.n_served}/{_inst.n}, doubles {_sol.n_double}/4")

_Z_pub = ILLUS["publie"][1].Z_complete
assert abs(_Z_pub - 70.82) < 5e-3, f"Table 1 publiée : Z* = {_Z_pub:.4f}, attendu 70,82"
print("Table 1 publiée (c_O = 0,5) : Z* = 70,82 retrouvé.")
if C_O_BASE == 0:
    _Z_hist = ILLUS["historique"][1].Z_complete
    assert abs(_Z_hist - 65.62) < 5e-3, f"Table 1 sans overtime : Z* = {_Z_hist:.4f}, attendu 65,62"
    print("Table 1 sans overtime (c_O = 0) : Z* = 65,62, valeur à reporter dans le manuscrit.")
# Les sections 3.2 à 6.2 utilisent la variante historique (c_O = C_O_BASE).
_inst3, _sol3 = ILLUS["historique"]


[publie] s_A=43.00, s_B=8.00 : statut optimal, gap 2.6e-15, Z* = 70.8200  (E[Idle]=59.150, E[Wait]=6.470, E[OT]=5.200), servis 7/7, doubles 3/4


[historique] s_A=43.00, s_B=8.00 : statut optimal, gap -2.2e-16, Z* = 65.6200  (E[Idle]=59.150, E[Wait]=6.470, E[OT]=0.000), servis 7/7, doubles 3/4


[donnees] s_A=36.00, s_B=7.00 : statut optimal, gap 8.6e-15, Z* = 79.7613  (E[Idle]=74.100, E[Wait]=5.661, E[OT]=0.000), servis 7/7, doubles 3/4
Table 1 publiée (c_O = 0,5) : Z* = 70,82 retrouvé.
Table 1 sans overtime (c_O = 0) : Z* = 65,62, valeur à reporter dans le manuscrit.


In [11]:
# ── 3.2 Test de validation Bernoulli sur les deux variantes ──────────────────
for _tag, (_inst, _sol) in ILLUS.items():
    _err = test_bernoulli_enum(_inst, _sol)
    print(f"[{_tag}] erreur max formule fermée / énumération : {_err:.2e}")


[publie] erreur max formule fermée / énumération : 3.55e-15
[historique] erreur max formule fermée / énumération : 3.55e-15
[donnees] erreur max formule fermée / énumération : 3.55e-15


In [12]:
# ── 3.3 Plannings et coefficients des paires (Table 1) ───────────────────────
def illustrative_tables(inst, sol):
    a, dl, w, o, gm = (sol.coeffs[k] for k in ("a", "delta", "w", "o", "gamma"))
    rows, pair_rows = [], []
    for (d, t) in sol.coeffs["slots"]:
        pats = sol.planning.get((d, t), [])
        ei, ew, eo = eval_slot_exact(pats, inst.q, inst.s, inst.L[d][t])
        # Cost columns are weighted by their unit cost: c_W E[Wait] is not minutes of waiting.
        rows.append({"Slot": f"S{t}", "L (min)": inst.L[d][t],
                     "Patients (SPT order)": ", ".join(map(str, pats)) or "-",
                     "Groups": "".join(ILLUS_GROUPS[i] for i in pats) or "-",
                     "E[Wait] (min)": ew, "c_I E[Idle]": inst.c_I * ei, "c_W E[Wait]": inst.c_W * ew,
                     "c_O E[OT]": inst.c_O * eo})
        if len(pats) == 2:
            i, j = min(pats), max(pats)
            pair_rows.append({"i": i, "j": j, "Slot": f"S{t}", "L (min)": inst.L[d][t],
                              "q_i": inst.q[i], "q_j": inst.q[j], "s_i": inst.s[i], "s_j": inst.s[j],
                              "delta": dl[i, j, d, t], "w": w[i, j], "gamma": gm[i, j, d, t],
                              "z coeff.": inst.c_W * w[i, j] - (inst.c_I + inst.c_O) * dl[i, j, d, t]})
    return pd.DataFrame(rows), pd.DataFrame(pair_rows)

ILLUS_TABLES = {}
for _tag, (_inst, _sol) in ILLUS.items():
    ILLUS_TABLES[_tag] = illustrative_tables(_inst, _sol)
    print(f"=== Table 1, variante {_tag} (Z* = {_sol.Z_complete:.2f}) ===")
    print(ILLUS_TABLES[_tag][0].round(3).to_string(index=False))
    print(ILLUS_TABLES[_tag][1].round(4).to_string(index=False) if len(ILLUS_TABLES[_tag][1])
          else "Aucune paire double.")
    print()


=== Table 1, variante publie (Z* = 70.82) ===
Slot  L (min) Patients (SPT order) Groups  E[Wait] (min)  c_I E[Idle]  c_W E[Wait]  c_O E[OT]
  S0     30.0                 3, 4     BB           4.20        18.40         2.10        0.0
  S1     30.0                    2      A           0.00         6.00         0.00        5.2
  S2     60.0                 6, 0     BA           4.32        16.50         2.16        0.0
  S3     60.0                 5, 1     BA           4.42        18.25         2.21        0.0
 i  j Slot  L (min)  q_i  q_j  s_i  s_j  delta    w  gamma  z coeff.
 3  4   S0     30.0 0.75 0.70  8.0  8.0    0.0 4.20   -0.0      2.10
 0  6   S2     60.0 0.90 0.60 43.0  8.0    0.0 4.32   -0.0      2.16
 1  5   S3     60.0 0.85 0.65 43.0  8.0    0.0 4.42   -0.0      2.21

=== Table 1, variante historique (Z* = 65.62) ===
Slot  L (min) Patients (SPT order) Groups  E[Wait] (min)  c_I E[Idle]  c_W E[Wait]  c_O E[OT]
  S0     30.0                    2      A           0.00       

 i  j Slot  L (min)  q_i  q_j  s_i  s_j  delta      w  gamma  z coeff.
 3  4   S1     30.0 0.75 0.70  7.0  7.0    0.0 3.6750   -0.0    1.8375
 1  5   S2     60.0 0.85 0.65 36.0  7.0    0.0 3.8675   -0.0    1.9338
 0  6   S3     60.0 0.90 0.60 36.0  7.0    0.0 3.7800   -0.0    1.8900



In [13]:
# ── 3.4 Export LaTeX (escaped, booktabs) ────────────────────────────────────
from data_driven_plan import latex_table
for _tag, (_df_plan, _df_pairs) in ILLUS_TABLES.items():
    _Z = ILLUS[_tag][1].Z_complete
    latex_table(_df_plan, f"tables/table1_planning_{_tag}.tex",
                caption=f"Illustrative schedule, {_tag} durations ($Z^\\star$ = {_Z:.2f}).",
                label=f"tab:planning-{_tag}", decimals=3)
    if len(_df_pairs):
        latex_table(_df_pairs, f"tables/table1_pairs_{_tag}.tex",
                    caption=f"Coefficients of the double-booked pairs at the optimum, {_tag} durations.",
                    label=f"tab:pairs-{_tag}", decimals=4)
print("Tables LaTeX exportées dans tables/")


Tables LaTeX exportées dans tables/


### Objectif résolu

La cellule suivante écrit, à partir des coûts effectivement passés au solveur, l'objectif du PLNE en LaTeX (`tables/objective_solved.tex`). Avec `C_O_BASE = 0` on retrouve l'équation (9) du manuscrit ; sinon le terme d'overtime apparaît explicitement.

In [14]:
_obj_tex = objective_latex(c_I=1.0, c_W=0.5, c_O=C_O_BASE)
with open("tables/objective_solved.tex", "w", encoding="utf-8") as fh:
    fh.write("% Objectif du PLNE tel que résolu (c_I=1, c_W=0.5, c_O=C_O_BASE)\n")
    fh.write("\\begin{equation}\n" + _obj_tex + "\n\\end{equation}\n")
from IPython.display import Math, display
display(Math(_obj_tex))
print(_obj_tex)


<IPython.core.display.Math object>

\min_{x,z}\; -1 \sum_{i,t} a_{it}\, x_{it} + \sum_{(i,j),t}\big(0{,}5\, w_{ij} - 1\, \delta_{ijt}\big)\, z_{ijt}


## 4. Générateur d'instances et plan d'expériences

Tailles, budgets et grilles viennent de `ilp_campaign.CampaignConfig` : production (une journée de S = 8 créneaux, R = 30 graines, gap cible 1e-6, limite de sécurité 600 s) ou smoke (S = 5, R = 2), modifiables par `CAMPAIGN_OVERRIDES`. Une journée est l'instance qu'un service planifie réellement ; elle est résolue à l'optimum prouvé en quelques secondes sur un ordinateur portable, alors que 5 jours × 8 créneaux (60 patients) gardaient un gap de 11 à 12 % après 1 800 s, sans amélioration de la solution entre 600 et 1 800 s. Plan OFAT autour de la cellule centrale (n/S = 1,5 ; p̄ = 0,20 ; part de créneaux longs 0,15 ; c_W/c_I = 0,50). Chaque PLNE est résolu une fois (cache disque), démarré à chaud depuis le glouton B3, avec un fil HiGHS par processus et un processus par cœur physique, les instances les plus chargées en premier (LPT).

In [15]:
# ── 4.1 Générateur (ilp_campaign.generate_instance) ──────────────────────────
# Correction P2 : q ~ Beta de moyenne 1 - p̄ et de concentration fixe kappa = 6,111
# (écart-type 0,15 à la cellule centrale p̄ = 0,20, comme avant), sans le plancher
# max(., 0.5) qui déplaçait la moyenne (0,844 au lieu de 0,90 à p̄ = 0,10).
import ilp_campaign as campaign
from ilp_campaign import BETA_CONCENTRATION, Q_CLIP, check_beta_mean, load_holdout_scores

def generate_instance(seed, ratio=1.5, p_bar=0.20, long_share=0.15, cW_ratio=0.5,
                      c_I=1.0, c_O=C_O_BASE, s_A_=None, s_B_=None, pi_A_=None,
                      Q_bar=None, q_pool=None) -> Instance:
    return campaign.generate_instance(
        seed, days=CC.days, slots_per_day=CC.slots_per_day, slot_lengths=SLOT_LENGTHS,
        ratio=ratio, p_bar=p_bar, long_share=long_share, cW_ratio=cW_ratio,
        s_A=s_A if s_A_ is None else s_A_, s_B=s_B if s_B_ is None else s_B_,
        pi_A=pi_A if pi_A_ is None else pi_A_, c_O=c_O, c_I=c_I, Q_bar=Q_bar, q_pool=q_pool)


In [16]:
# ── 4.1b Contrôle de la loi de q à chaque niveau de p̄ (10^5 tirages) ────────
for _pb in (0.03, 0.10, 0.20, 0.30):
    _m = check_beta_mean(_pb, n=100_000, seed=RANDOM_STATE)
    print(f"p̄ = {_pb:.2f} : moyenne empirique de q = {_m:.4f} (attendu {1-_pb:.4f}, |écart| < 0,01)")
print(f"kappa = {BETA_CONCENTRATION}, q tronqué à {Q_CLIP}")

# Variante empirique : q_i calibrés du holdout (mêmes grilles et groupes, q rééchantillonnés)
if HOLDOUT_SCORES_PATH is None:
    raise ValueError("HOLDOUT_SCORES_PATH requis : holdout_scores.parquet exporté par data_driven_plan.ipynb")
HOLDOUT = load_holdout_scores(HOLDOUT_SCORES_PATH)
Q_POOL = HOLDOUT["q_i"].to_numpy()
print(f"Holdout : {len(HOLDOUT):,} patients, q̄ = {Q_POOL.mean():.4f}, "
      f"no-show observé = {HOLDOUT['no_show'].mean():.4f}")
# La calibration isotone est en escalier : ses paliers extrêmes peuvent valoir 0 ou 1.
# Les q_i sont stockés tels que prédits et tronqués à Q_CLIP à l'entrée d'une instance.
Q_OUTSIDE = int(((Q_POOL < Q_CLIP[0]) | (Q_POOL > Q_CLIP[1])).sum())
print(f"q_i hors de {Q_CLIP} (tronqués dans les instances) : {Q_OUTSIDE} sur {len(Q_POOL):,} "
      f"({100 * Q_OUTSIDE / len(Q_POOL):.4f} %)")


p̄ = 0.03 : moyenne empirique de q = 0.9698 (attendu 0.9700, |écart| < 0,01)
p̄ = 0.10 : moyenne empirique de q = 0.8999 (attendu 0.9000, |écart| < 0,01)
p̄ = 0.20 : moyenne empirique de q = 0.7993 (attendu 0.8000, |écart| < 0,01)
p̄ = 0.30 : moyenne empirique de q = 0.6994 (attendu 0.7000, |écart| < 0,01)
kappa = 6.111, q tronqué à (0.01, 0.999)


Holdout : 4,013,341 patients, q̄ = 0.9684, no-show observé = 0.0317
q_i hors de (0.01, 0.999) (tronqués dans les instances) : 2073 sur 4,013,341 (0.0517 %)


In [17]:
# ── 4.2 Configuration de la campagne ─────────────────────────────────────────
import dataclasses, functools, json
from ilp_campaign import (CampaignConfig, CampaignContext, doe_specs, run_lpt, solve_spec,
                          simulate_spec, spec_cost)

CC = (CampaignConfig.smoke if SMOKE else CampaignConfig.production)(n_workers=N_WORKERS)
CC = dataclasses.replace(CC, **CAMPAIGN_OVERRIDES)
# Les q_i du holdout voyagent vers chaque processus : on en garde au plus 100 000,
# tirés une fois pour toutes (la loi rééchantillonnée reste celle du holdout).
_rng_pool = np.random.default_rng(RANDOM_STATE)
Q_POOL_SAMPLE = Q_POOL if len(Q_POOL) <= 100_000 else _rng_pool.choice(Q_POOL, 100_000, replace=False)
CTX = CampaignContext(cc=CC, s_A=s_A, s_B=s_B, pi_A=pi_A, c_O=C_O_BASE,
                      slot_lengths=tuple(float(v) for v in SLOT_LENGTHS),
                      cache_root=CACHE_DIR, q_pool=Q_POOL_SAMPLE)
R, SEEDS = CC.R, CC.seeds
with open("results/ilp_campaign_config.json", "w", encoding="utf-8") as fh:
    json.dump({"campaign": CC.as_dict(), "decisions": DECISIONS, "plne_parameters": PLNE},
              fh, indent=2, default=str)
for _k, _v in CC.as_dict().items():
    print(f"{_k:<20} {_v}")


days                 1
slots_per_day        8
R                    30
time_limit           600.0
mip_gap              1e-06
M_sim                10000
ratios               (1.0, 1.25, 1.5, 1.75, 2.0)
p_bars               (0.03, 0.1, 0.2, 0.3)
long_shares          (0.15, 0.25, 0.5)
cw_ratios            (0.1, 0.25, 0.5, 1.0, 2.0)
holdout_days         100
n_workers            4
seed_base            1000
central_ratio        1.5
central_p_bar        0.2
central_long_share   0.15
central_cw_ratio     0.5


In [18]:
# ── 4.3 Exécution du plan d'expériences ──────────────────────────────────────
DOE_SPECS = doe_specs(CC, with_holdout_q=True)
_t0_doe = time.perf_counter()
doe_results = run_lpt(functools.partial(solve_spec, CTX), DOE_SPECS, spec_cost, CC.n_workers)
_t_doe = time.perf_counter() - _t0_doe
_df_doe = pd.DataFrame(doe_results)
_df_doe["occ"] = _df_doe["n_served"] / _df_doe["n"]
_df_doe["double_rate"] = _df_doe["n_double"] / CC.S
_df_doe.to_csv("results/doe_ilp.csv", index=False)
print(f"{len(DOE_SPECS)} PLNE en {_t_doe/60:.1f} min sur {CC.n_workers} processus")
_doe_summary = _df_doe.groupby("label")[["n", "Z_complete", "E_idle", "E_wait", "E_OT", "n_served",
                                         "double_rate", "solve_time"]].mean()
print(_doe_summary.round(3))
# E_idle, E_wait, E_OT are weighted by c_I, c_W, c_O (cost units, not minutes)
_DOE_COLS = {"label": "Cell", "Z_complete": "Z", "E_idle": "c_I E[Idle]", "E_wait": "c_W E[Wait]",
             "E_OT": "c_O E[OT]", "n_served": "Served", "double_rate": "Double share", "solve_time": "Time (s)"}
_doe_tab = _doe_summary.reset_index().rename(columns=_DOE_COLS)
if C_O_BASE == 0:
    _doe_tab = _doe_tab.drop(columns="c_O E[OT]")   # identically 0 without an overtime cost
latex_table(_doe_tab, "tables/table_doe.tex",
            caption=f"Design of experiments: means per cell over R = {R} seeds. Cost columns are "
                    "weighted by their unit cost; every patient is served.", label="tab:doe", decimals=3)


450 PLNE en 0.2 min sur 4 processus
               n  Z_complete   E_idle  E_wait  E_OT  n_served  double_rate  \
label                                                                        
cW_0.1      12.0     128.033  126.224   1.809   0.0      12.0         0.50   
cW_0.25     12.0     130.746  126.224   4.522   0.0      12.0         0.50   
cW_1.0      12.0     144.283  126.775  17.507   0.0      12.0         0.50   
cW_2.0      12.0     161.217  127.708  33.509   0.0      12.0         0.50   
central     12.0     135.268  126.224   9.043   0.0      12.0         0.50   
ls_0.25     12.0     159.835  150.986   8.849   0.0      12.0         0.50   
ls_0.5      12.0     213.430  205.075   8.355   0.0      12.0         0.50   
pbar_0.03   12.0     113.996  101.111  12.885   0.0      12.0         0.50   
pbar_0.1    12.0     122.572  111.339  11.233   0.0      12.0         0.50   
pbar_0.3    12.0     150.145  143.342   6.803   0.0      12.0         0.50   
q_holdout   12.0     113.779

'\\begin{table}[htbp]\n\\caption{Design of experiments: means per cell over R = 30 seeds. Cost columns are weighted by their unit cost; every patient is served.}\n\\label{tab:doe}\n\\begin{tabular}{lrrrrrrr}\n\\toprule\nCell & n & Z & c\\_I E[Idle] & c\\_W E[Wait] & Served & Double share & Time (s) \\\\\n\\midrule\ncW\\_0.1 & 12.000 & 128.033 & 126.224 & 1.809 & 12.000 & 0.500 & 10.478 \\\\\ncW\\_0.25 & 12.000 & 130.746 & 126.224 & 4.522 & 12.000 & 0.500 & 12.976 \\\\\ncW\\_1.0 & 12.000 & 144.283 & 126.775 & 17.507 & 12.000 & 0.500 & 33.103 \\\\\ncW\\_2.0 & 12.000 & 161.217 & 127.708 & 33.509 & 12.000 & 0.500 & 39.505 \\\\\ncentral & 12.000 & 135.268 & 126.224 & 9.043 & 12.000 & 0.500 & 19.397 \\\\\nls\\_0.25 & 12.000 & 159.835 & 150.986 & 8.849 & 12.000 & 0.500 & 24.153 \\\\\nls\\_0.5 & 12.000 & 213.430 & 205.075 & 8.355 & 12.000 & 0.500 & 47.224 \\\\\npbar\\_0.03 & 12.000 & 113.996 & 101.111 & 12.885 & 12.000 & 0.500 & 24.431 \\\\\npbar\\_0.1 & 12.000 & 122.572 & 111.339 & 11.233 & 1

In [19]:
# ── 4.4 Statut de terminaison et gap (audit B1) ──────────────────────────────
print("Statuts HiGHS :", _df_doe["status"].value_counts().to_dict())
_unproven = _df_doe[_df_doe["status"] != "optimal"]
print(f"Optimalité prouvée : {100*(1-len(_unproven)/len(_df_doe)):.1f} % des PLNE")
print(f"Temps de résolution (s) : médiane {_df_doe['solve_time'].median():.1f}, "
      f"95e centile {_df_doe['solve_time'].quantile(0.95):.1f}, max {_df_doe['solve_time'].max():.1f}")
print(f"Mémoire résidente maximale d'un processus de résolution : {_df_doe['peak_rss_mb'].max():.0f} Mo")
if len(_unproven):
    print("Gap (coût complet) des solutions non prouvées : "
          f"médiane {_unproven['gap_complete'].median():.2e}, max {_unproven['gap_complete'].max():.2e}")
    print(_unproven.groupby("label").size().rename("non prouvés").to_string())


Statuts HiGHS : {'optimal': 449, 'maxTimeLimit': 1}
Optimalité prouvée : 99.8 % des PLNE
Temps de résolution (s) : médiane 21.2, 95e centile 125.0, max 600.6
Mémoire résidente maximale d'un processus de résolution : 597 Mo
Gap (coût complet) des solutions non prouvées : médiane 1.41e-02, max 1.41e-02
label
ratio_2.0    1


In [20]:
# ── 4.5 Figures du plan d'expériences (moyenne et IC à 95 %) ────────────────
from scipy import stats as sps

def _mean_ci(x):
    x = np.asarray(x, float)
    h = sps.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
    return x.mean(), h

_FACTORS = [("ratio", "Load n/S", "ratio_", CC.central_ratio),
            ("p_bar", "Mean no-show rate", "pbar_", CC.central_p_bar),
            ("long_share", "Share of long slots", "ls_", CC.central_long_share),
            ("cW_ratio", "$c_W / c_I$", "cW_", CC.central_cw_ratio)]
_METRICS = [("Z_complete", "Expected cost Z"), ("E_idle", "$c_I\\,E[\\mathrm{Idle}]$"),
            ("E_wait", "$c_W\\,E[\\mathrm{Wait}]$")]
if C_O_BASE != 0:
    _METRICS.append(("E_OT", "$c_O\\,E[\\mathrm{OT}]$"))   # identically 0 without an overtime cost
fig, axes = plt.subplots(len(_METRICS), len(_FACTORS), figsize=(7.2, 1.8 * len(_METRICS) + 0.3),
                         sharex="col")
for c, (fac, xlabel, prefix, center) in enumerate(_FACTORS):
    sub = _df_doe[(_df_doe["label"] == "central") | _df_doe["label"].str.startswith(prefix)]
    for r, (met, ylabel) in enumerate(_METRICS):
        g = sub.groupby(fac)[met].apply(_mean_ci)
        lv = np.array(g.index, float); mu = np.array([v[0] for v in g]); hw = np.array([v[1] for v in g])
        ax = axes[r, c]
        ax.errorbar(lv, mu, yerr=hw, fmt="o-", capsize=2)
        ax.grid(True, alpha=0.3)
        if c == 0:
            ax.set_ylabel(ylabel)
        if r == len(_METRICS) - 1:
            ax.set_xlabel(xlabel)
plt.tight_layout()
plt.savefig("figures/doe_metrics.pdf", bbox_inches="tight")
plt.savefig("figures/doe_metrics.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Benchmarks B1 à B5

Cellule centrale, R graines appariées. Le PLNE, B2, B3 et B5 servent tous les patients ; B1 (un patient par créneau) n'en sert que S et B4 est limité par son plafond de présence : leurs coûts portent sur moins de patients (colonne « Served »). B1 réservation simple (même objectif, un patient par créneau) ; B2 approximation fluide (Jensen) ; B3 glouton ; B4 utilité de type Valenzuela (plafond de présence espérée 1,2 par créneau) ; B5 surréservation uniforme (rotation jusqu'à deux patients par créneau, sans regarder q ni s ; rec. 8). Chaque planning est évalué avec le coût exact du modèle, puis rejoué en simulation avec les mêmes tirages pour toutes les politiques (nombres aléatoires communs). Écart au PLNE : moyenne, IC à 95 % de Student, test des rangs signés de Wilcoxon apparié.

In [21]:
# ── 5.1 Exécution ────────────────────────────────────────────────────────────
from ilp_campaign import BENCHMARKS, bench_spec, cell_spec, paired_summary

BENCH_SPECS = [cell_spec(CC, "central", sd) for sd in SEEDS]
_df_bench = pd.DataFrame(run_lpt(functools.partial(bench_spec, CTX), BENCH_SPECS,
                                 spec_cost, CC.n_workers))
_df_bench.to_csv("results/benchmarks.csv", index=False)
print("Statut du PLNE par graine :", _df_bench["ILP_status"].value_counts().to_dict())


Statut du PLNE par graine : {'optimal': 30}


In [22]:
# ── 5.2 Comparaison appariée au PLNE ─────────────────────────────────────────
_rows = []
for _b, (_name, _) in BENCHMARKS.items():
    for _kind, _suffix in (("exact model", "_Z"), ("simulated, CRN", "_Zsim")):
        _ps = paired_summary(_df_bench[_b + _suffix], _df_bench["ILP" + _suffix])
        _rows.append({"Policy": f"{_b} {_name}", "Cost": _kind,
                      "Mean ΔZ": _ps["mean_diff"], "95 % CI low": _ps["ci_low"],
                      "95 % CI high": _ps["ci_high"], "Mean gap (%)": _ps["rel_pct"],
                      "Beats the ILP": f"{_ps['n_better']}/{_ps['n']}", "p (Wilcoxon)": _ps["p_wilcoxon"],
                      "Doubles": _df_bench[f"{_b}_double"].mean(),
                      "Served": _df_bench[f"{_b}_served"].mean()})
_df_bench_cmp = pd.DataFrame(_rows)
print(_df_bench_cmp.round(3).to_string(index=False))
print(f"PLNE : doubles moyens {_df_bench['ILP_double'].mean():.1f}, servis {_df_bench['ILP_served'].mean():.1f}, "
      f"coût exact moyen {_df_bench['ILP_Z'].mean():.2f}")
latex_table(_df_bench_cmp, "tables/table_benchmarks.tex",
            caption=f"Benchmarks against the ILP, central cell, R = {R} paired seeds. The ILP serves "
                    f"every patient ({_df_bench['ILP_served'].mean():.0f}); B1 and B4 serve fewer.",
            label="tab:benchmarks", decimals=3)
# Only a policy serving every patient competes on the ILP's own feasible set
_full = _df_bench_cmp["Served"] >= _df_bench["ILP_served"].mean() - 1e-9
_beaten = _df_bench_cmp[(_df_bench_cmp["Cost"] == "exact model") & _full
                        & (_df_bench_cmp["Beats the ILP"].str.split("/").str[0] != "0")]
if len(_beaten):
    print("ATTENTION : une politique bat le PLNE sous son propre coût, donc au moins une "
          "solution du PLNE n'est pas optimale (voir les statuts ci-dessus).")


                Policy           Cost  Mean ΔZ  95 % CI low  95 % CI high  Mean gap (%) Beats the ILP  p (Wilcoxon)  Doubles  Served
     B1 Single booking    exact model   10.660        9.878        11.441         7.986          0/30         0.000    0.000   8.000
     B1 Single booking simulated, CRN    6.263        2.622         9.903         4.152          3/30         0.000    0.000   8.000
     B2 Fluid (Jensen)    exact model    5.468        3.573         7.364         4.409          0/30         0.000    4.500  12.000
     B2 Fluid (Jensen) simulated, CRN    2.640        0.872         4.409         2.032          6/30         0.002    4.500  12.000
             B3 Greedy    exact model    3.503        2.373         4.633         2.675          0/30         0.000    4.000  12.000
             B3 Greedy simulated, CRN    4.701        2.131         7.270         3.953          3/30         0.000    4.000  12.000
 B4 Valenzuela utility    exact model   32.097       26.289        37

## 6. Simulation hors hypothèse d'indépendance, dynamique séquentielle (Lindley)

Chaque planning optimal est rejoué sur une journée réelle : retard propagé d'un créneau au suivant, idle terminal compté (correction P1). M scénarios par planning (`CC.M_sim`).

In [23]:
# ── 6.1 Simulateur Lindley (data_driven_plan.simulate_day / simulate_instance) ─
# Correction P1 : le simulateur compte désormais l'idle terminal, entre la fin du
# dernier patient et la fin de la journée ; sans lui, une journée qui finit tôt
# paraissait gratuite. simulate_instance renvoie le coût de chacun des M scénarios
# (moyenne, IC et tests appariés possibles) ; eval_day_exact donne la même
# espérance exactement, par énumération des 2^n profils de présence d'une journée.
print("simulate_day, simulate_instance, eval_day_exact importés de data_driven_plan")


simulate_day, simulate_instance, eval_day_exact importés de data_driven_plan


In [24]:
# ── 6.2 Validation sur l'exemple illustratif ─────────────────────────────────
# Trois évaluations du même planning optimal :
#  - modèle      : créneaux indépendants, ce que le PLNE optimise (eval_slot_exact) ;
#  - jour exact  : dynamique de Lindley avec report et idle terminal, énumération 2^n ;
#  - simulé      : même dynamique, M scénarios ; doit retrouver le jour exact à 4 erreurs-types près.
_rng6 = np.random.default_rng(RANDOM_STATE)
_M_ILLUS = 10_000 if SMOKE else 200_000
_model_Z3 = evaluate_planning_exact(_inst3, _sol3.planning, day_dynamics=False)["Z"]
_exact_Z3 = evaluate_planning_exact(_inst3, _sol3.planning, day_dynamics=True)["Z"]
_sim3 = simulate_instance(_inst3, _sol3.planning, _M_ILLUS, _rng6, cv=0.0)
_se3 = _sim3.std(ddof=1) / np.sqrt(_M_ILLUS)
print(f"Exemple illustratif (M={_M_ILLUS:,}) :")
print(f"  Coût modèle (créneaux indépendants) : {_model_Z3:.4f}")
print(f"  Coût exact de la journée (Lindley)   : {_exact_Z3:.4f}")
print(f"  Coût simulé                          : {_sim3.mean():.4f} ± {1.96*_se3:.4f} (IC 95 %)")
assert abs(_sim3.mean() - _exact_Z3) < 4 * _se3, "simulateur et évaluation exacte divergent"
print(f"  Écart jour/modèle : {100*(_exact_Z3/_model_Z3-1):+.2f} % (erreur de l'hypothèse d'indépendance A6)")


Exemple illustratif (M=200,000) :
  Coût modèle (créneaux indépendants) : 65.6200
  Coût exact de la journée (Lindley)   : 62.7600
  Coût simulé                          : 62.7379 ± 0.1028 (IC 95 %)
  Écart jour/modèle : -4.36 % (erreur de l'hypothèse d'indépendance A6)


In [25]:
# ── 6.3 Écart modèle / simulé sur le plan d'expériences ─────────────────────
# Les plannings viennent du cache de la section 4 (aucune nouvelle résolution).
# Durées déterministes (cv = 0) puis lognormales (cv = 0,3 et 0,5), même moyenne.
M_DOE_SIM = CC.M_sim
_t0_sim = time.perf_counter()
sim_results = run_lpt(functools.partial(simulate_spec, CTX), DOE_SPECS, spec_cost, CC.n_workers)
print(f"Simulation de {len(DOE_SPECS)} plannings (M={M_DOE_SIM:,}) en {(time.perf_counter()-_t0_sim)/60:.1f} min")
_df_sim = pd.DataFrame(sim_results)
_df_sim.to_csv("results/simulation_doe.csv", index=False)
print("Écart % simulé (cv = 0) / modèle, par cellule :")
print(_df_sim.groupby("label")["gap_pct"].agg(["mean", "std"]).round(2))


Simulation de 450 plannings (M=10,000) en 0.1 min
Écart % simulé (cv = 0) / modèle, par cellule :
             mean    std
label                   
cW_0.1      -7.62   5.68
cW_0.25     -3.72   5.13
cW_1.0      13.17  37.00
cW_2.0      32.88  67.62
central      2.29  15.19
ls_0.25      0.38   6.40
ls_0.5      -0.20   0.47
pbar_0.03   19.69  93.24
pbar_0.1     9.01  45.55
pbar_0.3    -0.55   4.41
q_holdout   14.47  64.88
ratio_1.0   -1.91   1.91
ratio_1.25  -0.81   2.89
ratio_1.75  12.38  24.36
ratio_2.0   46.66  74.39


In [26]:
# ── 6.4 Figures : écart % selon la charge et la part de doubles ─────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 2.8))
_sub = _df_sim[(_df_sim["label"] == "central") | _df_sim["label"].str.startswith("ratio_")]
g = _sub.groupby("ratio")["gap_pct"].apply(_mean_ci)
ax1.errorbar(g.index, [v[0] for v in g], yerr=[v[1] for v in g], fmt="o-", capsize=2)
ax1.set_xlabel("Load n/S"); ax1.set_ylabel("Simulated vs model cost (%)")
ax1.grid(True, alpha=0.3)
ax2.scatter(_df_sim["double_rate"], _df_sim["gap_pct"], alpha=0.4, s=8)
ax2.set_xlabel("Share of double-booked slots"); ax2.set_ylabel("Simulated vs model cost (%)")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/sim_gap_analysis.pdf", bbox_inches="tight")
plt.savefig("figures/sim_gap_analysis.png", dpi=300, bbox_inches="tight")
plt.show()


In [27]:
# ── 6.5 Variabilité des durées (lognormale, cv = 0,3 et 0,5) ────────────────
fig, ax = plt.subplots(figsize=(4.2, 2.8))
for cv in (0.0, 0.3, 0.5):
    col = f"Z_sim_cv{cv:g}"
    dev = 100 * (_sub[col] / _sub["Z_model"] - 1)
    g = dev.groupby(_sub["ratio"]).apply(_mean_ci)
    ax.errorbar(g.index, [v[0] for v in g], yerr=[v[1] for v in g], fmt="o-", capsize=2,
                label=f"cv = {cv:g}")
ax.set_xlabel("Load n/S"); ax.set_ylabel("Simulated vs model cost (%)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/sim_lognormal.pdf", bbox_inches="tight")
plt.savefig("figures/sim_lognormal.png", dpi=300, bbox_inches="tight")
plt.show()


## 7. Analyse de sensibilité et robustesse

Correction P3. Pour chaque perturbation, le PLNE est résolu sur l'instance perturbée (ce que verrait l'optimiseur), puis son planning est évalué sur l'instance vraie et comparé à l'optimum vrai (même cellule, même graine, solution du cache). Évaluation **exacte** (coût du modèle) et **simulée** avec des nombres aléatoires communs aux deux plannings. Les perturbations d'un même type partagent leurs tirages d'un niveau à l'autre. La mauvaise classification va jusqu'au taux d'erreur mesuré du classifieur de groupe. Aucune conclusion comparative n'est tirée sans test apparié significatif (Wilcoxon, 5 %).

In [28]:
# ── 7.1 Regret sous perturbation des entrées ─────────────────────────────────
from ilp_campaign import sensitivity_spec, sensitivity_specs, sensitivity_table

SENS_SPECS = sensitivity_specs(CC, PLNE["group_error_rate"])
_df_sens_raw = pd.DataFrame(run_lpt(functools.partial(sensitivity_spec, CTX), SENS_SPECS,
                                    spec_cost, CC.n_workers))
_df_sens_raw.to_csv("results/sensitivity_raw.csv", index=False)
_df_sens = sensitivity_table(_df_sens_raw)
_df_sens.to_csv("results/sensitivity_summary.csv", index=False)
print(_df_sens.round(4).to_string(index=False))


Perturbation   Level  n  Exact regret (%)  Exact ΔZ  95 % CI low  95 % CI high  Exact p  Simulated ΔZ  Simulated p  Significant (5 %)
    duration -0.2000 30            0.5414    0.6985       0.5355        0.8615   0.0000        1.2711       0.0111               True
    duration -0.1000 30            0.3458    0.4264       0.2521        0.6007   0.0001        0.9056       0.1836               True
    duration  0.1000 30           -0.0001   -0.0001      -0.0003        0.0001   0.1573       -0.3324       0.9569              False
    duration  0.2000 30           -0.0001   -0.0001      -0.0003        0.0001   0.5271       -0.0861       0.7731              False
    misclass  0.0500 30            2.4722    2.8377       1.0164        4.6590   0.0010        3.1396       0.0035               True
    misclass  0.1000 30            3.5625    4.2449       2.2171        6.2726   0.0000        3.7611       0.0027               True
    misclass  0.2000 30            5.6275    6.5114       4.32

In [29]:
# ── 7.2 Figure : regret exact par perturbation (moyenne et IC à 95 %) ───────
_KIND_LABEL = {"q_shift": "Shift of q", "q_noise": "Gaussian noise on q (sd)",
               "duration": "Relative error on s", "misclass": "Group misclassification rate"}
fig, axes = plt.subplots(2, 2, figsize=(6.0, 4.4))
for ax, (kind, lab) in zip(axes.flat, _KIND_LABEL.items()):
    sub = _df_sens_raw[_df_sens_raw["kind"] == kind]
    g = sub.groupby("level")["deg_pct"].apply(_mean_ci)
    ax.errorbar(g.index, [v[0] for v in g], yerr=[v[1] for v in g], fmt="o-", capsize=2)
    ax.axhline(0, color="grey", lw=0.8)
    if len(g) == 1:  # un seul niveau : un seul repère, centré
        ax.set_xticks(g.index); ax.set_xlim(g.index[0] - 0.05, g.index[0] + 0.05)
    ax.set_xlabel(lab); ax.set_ylabel("Exact regret (%)"); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/sensitivity_regret.pdf", bbox_inches="tight")
plt.savefig("figures/sensitivity_regret.png", dpi=300, bbox_inches="tight")
plt.show()


In [30]:
# ── 7.3 Plafond de présence Q̄ et part du groupe A (10 premières graines) ────
Q_BARS = [1.0, 1.2, 1.5, None]
PI_AS = sorted({0.05, 0.15, 0.30, round(pi_A, 4)})
QBAR_SPECS = [cell_spec(CC, "qbar", sd, Q_bar=qb, pi_A=pa)
              for qb in Q_BARS for pa in PI_AS for sd in SEEDS[:10]]
# Tous les patients étant servis, un plafond Q̄ trop bas rend l'instance infaisable
# (somme des q_i > S Q̄) : HiGHS le prouve, la cellule est comptée « infeasible », sans coût.
_df_qbar = pd.DataFrame(run_lpt(functools.partial(solve_spec, CTX, allow_infeasible=True),
                                QBAR_SPECS, spec_cost, CC.n_workers))
_df_qbar["Q_bar"] = _df_qbar["Q_bar"].map(lambda v: "none" if pd.isna(v) else f"{v:g}")
_df_qbar.to_csv("results/sensitivity_qbar.csv", index=False)
print("Statuts :", _df_qbar["status"].value_counts().to_dict())
print("Instances infaisables par Q̄ :")
print(_df_qbar.assign(inf=_df_qbar["status"] == "infeasible").pivot_table(
    values="inf", index="Q_bar", columns="pi_A", aggfunc="sum"))
print("Coût moyen des instances faisables :")
print(_df_qbar.pivot_table(values="Z_complete", index="Q_bar", columns="pi_A", aggfunc="mean").round(3))


xgboost 3.2.0 | imbalanced-learn 0.14.2
xgboost 3.2.0 | imbalanced-learn 0.14.2
xgboost 3.2.0 | imbalanced-learn 0.14.2
xgboost 3.2.0 | imbalanced-learn 0.14.2


Statuts : {'infeasible': 104, 'optimal': 56}
Instances infaisables par Q̄ :
pi_A   0.0500  0.1500  0.3000  0.3463
Q_bar                                
1          10      10      10      10
1.2        10      10      10      10
1.5         6       6       6       6
none        0       0       0       0
Coût moyen des instances faisables :
pi_A    0.0500   0.1500   0.3000   0.3463
Q_bar                                    
1.5    203.367  173.929  136.297  129.344
none   196.235  178.879  134.353  126.293


In [31]:
# ── 7.4 Front idle / attente (balayage de c_W/c_I, 10 premières graines) ────
CW_SWEEP = [0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
PARETO_SPECS = [cell_spec(CC, "pareto", sd, cW_ratio=v) for v in CW_SWEEP for sd in SEEDS[:10]]
_df_pareto = pd.DataFrame(run_lpt(functools.partial(solve_spec, CTX), PARETO_SPECS, spec_cost, CC.n_workers))
# E_wait est pondéré par c_W : on revient aux minutes d'attente pour le front
_df_pareto["wait_min"] = _df_pareto["E_wait"] / _df_pareto["cW_ratio"]
_df_pareto.to_csv("results/pareto_idle_wait.csv", index=False)
grp_p = _df_pareto.groupby("cW_ratio")[["E_idle", "wait_min"]].mean()
fig, ax = plt.subplots(figsize=(4.4, 3.4))
sc = ax.scatter(grp_p["E_idle"], grp_p["wait_min"], c=np.log(grp_p.index), cmap="viridis", s=25, zorder=5)
# Plusieurs ratios peuvent donner les mêmes plannings : une seule étiquette par point
for (_x, _y), _g in grp_p.round(6).reset_index().groupby(["E_idle", "wait_min"], sort=False):
    ax.annotate(", ".join(f"{v:g}" for v in _g["cW_ratio"]), (_x, _y), fontsize=6,
                textcoords="offset points", xytext=(4, 3))
ax.plot(grp_p["E_idle"], grp_p["wait_min"], "k--", alpha=0.4)
ax.set_xlabel("Expected idle time E[Idle] (min)"); ax.set_ylabel("Expected waiting time E[Wait] (min)")
ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label="$\\log(c_W / c_I)$")
plt.tight_layout()
plt.savefig("figures/pareto_idle_wait.pdf", bbox_inches="tight")
plt.savefig("figures/pareto_idle_wait.png", dpi=300, bbox_inches="tight")
plt.show()


In [32]:
# ── 7.5 Synthèse : seules les différences significatives sont qualifiées ────
_sig = _df_sens[_df_sens["Significant (5 %)"]]
_nsig = _df_sens[~_df_sens["Significant (5 %)"]]
print(f"{len(_sig)} perturbation(s) sur {len(_df_sens)} dégradent significativement le coût (Wilcoxon, 5 %).")
for _, r in _sig.iterrows():
    print(f"  {r['Perturbation']} {r['Level']:+g} : regret {r['Exact regret (%)']:.2f} %, "
          f"IC 95 % de ΔZ [{r['95 % CI low']:.2f} ; {r['95 % CI high']:.2f}], p = {r['Exact p']:.3g}")
if len(_nsig):
    print("Non significatif (aucune conclusion) : " +
          ", ".join(f"{r['Perturbation']} {r['Level']:+g}" for _, r in _nsig.iterrows()))
latex_table(_df_sens, "tables/table_sensitivity.tex",
            caption="Regret under perturbed inputs (95 % CI, paired Wilcoxon tests).",
            label="tab:sensitivity", decimals=4)
SENS_VERDICT = (f"{len(_sig)}/{len(_df_sens)} perturbations significatives ; regret exact maximal "
                f"{_df_sens['Exact regret (%)'].max():.2f} %")


8 perturbation(s) sur 13 dégradent significativement le coût (Wilcoxon, 5 %).
  duration -0.2 : regret 0.54 %, IC 95 % de ΔZ [0.54 ; 0.86], p = 2.56e-06
  duration -0.1 : regret 0.35 %, IC 95 % de ΔZ [0.25 ; 0.60], p = 9.74e-05
  misclass +0.05 : regret 2.47 %, IC 95 % de ΔZ [1.02 ; 4.66], p = 0.000982
  misclass +0.1 : regret 3.56 %, IC 95 % de ΔZ [2.22 ; 6.27], p = 1.82e-05
  misclass +0.2 : regret 5.63 %, IC 95 % de ΔZ [4.33 ; 8.70], p = 2.56e-06
  misclass +0.3323 : regret 9.08 %, IC 95 % de ΔZ [5.98 ; 14.03], p = 2.56e-06
  q_noise +0.05 : regret 0.10 %, IC 95 % de ΔZ [0.08 ; 0.17], p = 1.86e-09
  q_shift +0.1 : regret 0.06 %, IC 95 % de ΔZ [0.04 ; 0.12], p = 7.98e-06
Non significatif (aucune conclusion) : duration +0.1, duration +0.2, q_shift -0.1, q_shift -0.05, q_shift +0.05


## 8. Validation hors échantillon sur des patients du holdout (rec. 10)

Journées construites à partir de patients du holdout, jamais vus à l'apprentissage : le PLNE planifie avec les $q_i$ et les groupes **prédits** ; chaque planning est ensuite jugé sur la vérité : groupes **réels** (durées), coût espéré du modèle, coût espéré exact de la journée (Lindley, idle terminal) et coût **réalisé** compte tenu des présences observées. Références : PLNE oracle (groupes réels), B1, B3, B5. Comparaisons appariées par journée. B1 ne sert que S patients sur n ; les autres politiques les servent tous.

In [33]:
# ── 8.1 Journées du holdout ──────────────────────────────────────────────────
from ilp_campaign import holdout_day, holdout_day_specs
HOLDOUT_SPECS = holdout_day_specs(CC, HOLDOUT, seed=RANDOM_STATE)
_df_hold = pd.DataFrame(run_lpt(functools.partial(holdout_day, CTX), HOLDOUT_SPECS,
                                lambda sp: 1.0, CC.n_workers))
_df_hold.to_csv("results/holdout_days.csv", index=False)
print(f"{len(_df_hold)} journées de {_df_hold['n'].iloc[0]} patients ; statut du PLNE :",
      _df_hold["ILP_status"].value_counts().to_dict())
_rows = []
for _ref in ("ILP_oracle", "B1", "B3", "B5"):
    for _kind, _suffix in (("expected, model", "_Zmodel"), ("expected, day", "_Zday"), ("realised", "_Zreal")):
        _ps = paired_summary(_df_hold[_ref + _suffix], _df_hold["ILP" + _suffix])
        _rows.append({"Policy": _ref, "Cost": _kind, "Mean ΔZ vs ILP": _ps["mean_diff"],
                      "95 % CI low": _ps["ci_low"], "95 % CI high": _ps["ci_high"],
                      "Mean gap (%)": _ps["rel_pct"], "Does better": f"{_ps['n_better']}/{_ps['n']}",
                      "p (Wilcoxon)": _ps["p_wilcoxon"], "Served": _df_hold[_ref + "_served"].mean()})
_df_hold_cmp = pd.DataFrame(_rows)
_df_hold_cmp.to_csv("results/holdout_days_summary.csv", index=False)
latex_table(_df_hold_cmp, "tables/table_holdout_days.tex",
            caption="Holdout days: paired differences with the ILP (95 % CI, Wilcoxon). B1 serves "
                    "one patient per slot, the other policies every patient.",
            label="tab:holdout", decimals=3)
print(_df_hold_cmp.round(3).to_string(index=False))


100 journées de 12 patients ; statut du PLNE : {'optimal': 100}
    Policy            Cost  Mean ΔZ vs ILP  95 % CI low  95 % CI high  Mean gap (%) Does better  p (Wilcoxon)  Served
ILP_oracle expected, model         -19.899      -23.259       -16.540       -17.042     100/100         0.000    12.0
ILP_oracle   expected, day         -24.878      -30.573       -19.184       -16.220      93/100         0.000    12.0
ILP_oracle        realised         -24.605      -30.788       -18.422       -15.691      88/100         0.000    12.0
        B1 expected, model          16.007       12.265        19.750        14.855      17/100         0.000     8.0
        B1   expected, day          -3.564      -11.992         4.863         1.178      38/100         0.507     8.0
        B1        realised          -4.755      -14.243         4.733         1.622      37/100         0.389     8.0
        B3 expected, model          -0.263       -3.517         2.990         0.625      50/100         0.628 

In [34]:
# ── 9. Export RESULTS.md ─────────────────────────────────────────────────────
_central = _df_doe[_df_doe.label == "central"]
_sol_hist = ILLUS["historique"][1]
def _cmp_lines(df, kind):
    return "\n".join(
        f"- {r['Policy']} : {r.iloc[2]:+.2f} [{r['95 % CI low']:.2f} ; {r['95 % CI high']:.2f}], "
        f"p = {r['p (Wilcoxon)']:.3g}, served {r['Served']:.1f}" for _, r in df[df["Cost"] == kind].iterrows())

_md = f"""# RESULTS.md, chiffres clés pour le manuscrit

Paramètres : {PLNE['source_run']} (commit {PLNE['git_commit']}), s_A = {s_A:.2f}, s_B = {s_B:.2f},
pi_A = {pi_A:.4f}, c_O = {C_O_BASE}. Campagne : S = {CC.S}, R = {CC.R}, limite {CC.time_limit:g} s,
gap cible {CC.mip_gap:g}. Smoke : {SMOKE}. Tous les patients servis (C1 en égalité).
Holdout : {len(Q_POOL):,} patients, {Q_OUTSIDE} q_i hors de {Q_CLIP}, tronqués dans les instances.

## Table 1 (section 3)
- Historique (n = {ILLUS['historique'][0].n}, s_A = 43, s_B = 8) : Z* = {_sol_hist.Z_complete:.4f} ({_sol_hist.status})
- Données : Z* = {ILLUS['donnees'][1].Z_complete:.4f}

## Plan d'expériences
- Optimalité prouvée : {100*(_df_doe['status'] == 'optimal').mean():.1f} % des PLNE
- Résolution : médiane {_df_doe['solve_time'].median():.1f} s, max {_df_doe['solve_time'].max():.1f} s,
  mémoire max {_df_doe['peak_rss_mb'].max():.0f} Mo par processus ({CPU_MODEL},
  {psutil.virtual_memory().total / 2**30:.0f} Go de RAM)
- Cellule centrale : Z = {_central['Z_complete'].mean():.3f} ± {_central['Z_complete'].std():.3f}

## Benchmarks (coût exact du modèle, Delta Z vs PLNE, IC 95 %)
{_cmp_lines(_df_bench_cmp, 'exact model')}

## Simulation (M = {M_DOE_SIM})
- Écart simulé / modèle, cellule centrale : {_df_sim[_df_sim.label == 'central']['gap_pct'].mean():.2f} %

## Robustesse
- {SENS_VERDICT}

## Hors échantillon (holdout, coût réalisé, Delta Z vs PLNE)
{_cmp_lines(_df_hold_cmp, 'realised')}
"""
with open("results/RESULTS.md", "w", encoding="utf-8") as fh:
    fh.write(_md)
print(_md)


# RESULTS.md, chiffres clés pour le manuscrit

Paramètres : run_full_20261030 (commit 8615e6564b67321c11e2ee6208425014f15d0c85), s_A = 36.00, s_B = 7.00,
pi_A = 0.3463, c_O = 0.0. Campagne : S = 8, R = 30, limite 600 s,
gap cible 1e-06. Smoke : False. Tous les patients servis (C1 en égalité).
Holdout : 4,013,341 patients, 2073 q_i hors de (0.01, 0.999), tronqués dans les instances.

## Table 1 (section 3)
- Historique (n = 7, s_A = 43, s_B = 8) : Z* = 65.6200 (optimal)
- Données : Z* = 79.7613

## Plan d'expériences
- Optimalité prouvée : 99.8 % des PLNE
- Résolution : médiane 21.2 s, max 600.6 s,
  mémoire max 597 Mo par processus (Intel(R) Core(TM) i7-6700HQ CPU @ 2.60GHz,
  15 Go de RAM)
- Cellule centrale : Z = 135.268 ± 27.046

## Benchmarks (coût exact du modèle, Delta Z vs PLNE, IC 95 %)
- B1 Single booking : +10.66 [9.88 ; 11.44], p = 1.86e-09, served 8.0
- B2 Fluid (Jensen) : +5.47 [3.57 ; 7.36], p = 1.86e-09, served 12.0
- B3 Greedy : +3.50 [2.37 ; 4.63], p = 1.86e-09, served